# Experiment 4.0 — Fixed250 temporal SNN decoder

## Scientific question

Can a stateful SNN replace a flattened Linear classifier after the raw 30-channel weighted event train has already been compressed into ordered 250 ms channel-wise sums?

All methods consume the same representation:

$$X_{1:T}\rightarrow z_1,\ldots,z_B,\qquad z_{b,c}=\sum_{t\in W_b,\ t<L}X_{t,c}.$$

A train-only per-channel scale is fitted on valid bins and applied without mean subtraction, so padded zero bins remain exactly zero.

The SNN always executes the complete padded macro sequence. Valid length changes only the readout window, not state evolution. Primary training uses valid WholeCount CE; full WholeCount and valid/full final membrane are diagnostic readouts from the same trajectory.

## Experiment matrix

- Decoder: `ff`, `rsnn`
- Hidden width: 32, 64, 128
- Hidden $\tau_{mem}$: 250, 500, 1000, 2000 ms
- Seeds: 11, 23, 37, 53, 71
- 120 independent SNN runs + one shared Fixed250+Linear baseline

For hidden width $H$, recurrence adds $H^2$ trainable parameters because $W_{rec}\in\mathbb{R}^{H\times H}$.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts import experiment_4_0_fixed250_temporal_snn as exp40

root = exp40.results_dir(repo_root)
runs = pd.read_csv(root / 'runs.csv')
summary = pd.read_csv(root / 'summary.csv')
baseline = json.loads((root / 'baseline' / 'fixed250_linear.json').read_text())
display(summary)
print('Linear test BA:', baseline['metrics']['test']['balanced_accuracy'])

In [ ]:
test = runs[runs['split'] == 'test'].copy()
mean_ba = (
    test.groupby(['architecture', 'hidden_width', 'tau_mem_ms'], as_index=False)
        ['valid_count_balanced_accuracy'].mean()
)
fig, ax = plt.subplots(figsize=(9, 5.5))
for (architecture, width), frame in mean_ba.groupby(['architecture', 'hidden_width']):
    frame = frame.sort_values('tau_mem_ms')
    ax.plot(frame['tau_mem_ms'], frame['valid_count_balanced_accuracy'], marker='o', label=f'{architecture.upper()} H={width}')
ax.axhline(baseline['metrics']['test']['balanced_accuracy'], linestyle='--', label='Fixed250 + Linear')
ax.set_xscale('log', base=2)
ax.set_xlabel(r'Hidden $\tau_{mem}$ (ms)')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp4.0: passive memory and recurrence')
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

In [ ]:
paired = (
    test.groupby(['architecture', 'hidden_width', 'tau_mem_ms', 'seed'], as_index=False)
        [['valid_count_balanced_accuracy', 'full_count_balanced_accuracy', 'tail_spike_fraction']].first()
)
display(paired.sort_values('valid_count_balanced_accuracy', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.scatter(paired['valid_count_balanced_accuracy'], paired['full_count_balanced_accuracy'])
lo = min(paired['valid_count_balanced_accuracy'].min(), paired['full_count_balanced_accuracy'].min())
hi = max(paired['valid_count_balanced_accuracy'].max(), paired['full_count_balanced_accuracy'].max())
ax.plot([lo, hi], [lo, hi], linestyle='--')
ax.set_xlabel('Valid WholeCount BA')
ax.set_ylabel('Full padded WholeCount BA')
ax.set_title('Endpoint readout vs post-end dynamics')
ax.grid(True, alpha=0.25)
plt.show()

In [ ]:
param_view = (
    test.groupby(['architecture', 'hidden_width', 'tau_mem_ms', 'parameter_count'], as_index=False)
        ['valid_count_balanced_accuracy'].mean()
)
fig, ax = plt.subplots(figsize=(8, 5.5))
for architecture, frame in param_view.groupby('architecture'):
    ax.scatter(frame['parameter_count'], frame['valid_count_balanced_accuracy'], label=architecture.upper())
ax.axhline(baseline['metrics']['test']['balanced_accuracy'], linestyle='--', label='Fixed250 + Linear')
ax.set_xlabel('Trainable parameters')
ax.set_ylabel('Mean test balanced accuracy')
ax.set_title('Parameter efficiency')
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()